# Finish b3 without Google Drive

b3 broke off after epoch 7. This notebook trains the last epoch from `checkpoint.pt` in any Colab account, without Drive. Choose Runtime, Change runtime type, T4 GPU, then Run all. It asks for the starter-photo link and for two files: `checkpoint.pt` and `mine.zip`.

In [ ]:
import subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise SystemExit("No GPU: Runtime, Change runtime type, T4 GPU, Save, then Run all again.")

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 onnx==1.19.1 onnxruntime==1.23.2 onnxscript

In [ ]:
import os, time
os.environ["TZ"] = "Asia/Seoul"; time.tzset()
!git clone -q https://github.com/wlsdid/mas1004-assignment2 /content/a2
os.chdir("/content/a2")
!git log -1 --format="code from commit %h: %s"

In [ ]:
import subprocess
link = input("The link to the starter photos, from Cyber Campus: ").strip()
subprocess.run(["python", "src/get_data.py", "--link", link], check=True)

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
print("Choose checkpoint.pt and mine.zip (both at once).")
got = files.upload()
Path("results/b3").mkdir(parents=True, exist_ok=True)
shutil.move("checkpoint.pt", "results/b3/checkpoint.pt")
zipfile.ZipFile("mine.zip").extractall("data")
print(sum(1 for p in Path("data/mine").rglob("*") if p.is_file()), "files in data/mine")

In [ ]:
!python src/train_b.py --tag b3 --model efficientnet_b3 --size 320 --epochs 8 --batch-size 32 --balance --no-flip --gpu-decode --folders data/starter/train data/mine --submit 2>&1 | tee b3_finish.txt
!python src/check_submission.py --no-val 2>&1 | tee -a b3_finish.txt

In [ ]:
!zip -q -r b3_finish.zip submission b3_finish.txt experiments.csv
from google.colab import files
files.download("b3_finish.zip")